In [1]:
from google.colab import files
import io

uploaded = files.upload()


Saving train.csv to train.csv


**Phase 1**

---



**Install & Import Pandera:**

---



In [11]:
!pip install pandera
import pandera as pa
from pandera import Column, DataFrameSchema, Check
print("Pandera loaded!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.5/442.5 kB 9.0 MB/s eta 0:00:00
Pandera loaded!


**Define the Schema:**

---



In [12]:
schema = DataFrameSchema(
    columns={
        # Target variable
        "SalePrice": Column(
            dtype=float,
            checks=[
                Check.greater_than(0),
                Check.less_than(800000)
            ],
            nullable=False
        ),

        # Engineered features validation
        "house_age": Column(
            dtype=float,
            checks=[
                Check.greater_than_or_equal_to(0),
                Check.less_than(200)
            ],
            nullable=False
        ),

        "total_bathrooms": Column(
            dtype=float,
            checks=[
                Check.greater_than_or_equal_to(0),
                Check.less_than_or_equal_to(10)
            ],
            nullable=False
        ),

        "total_area": Column(
            dtype=float,
            checks=[
                Check.greater_than(0),
                Check.less_than(15000)
            ],
            nullable=False
        ),

        "quality_score": Column(
            dtype=float,
            checks=[
                Check.greater_than(0),
                Check.less_than_or_equal_to(100)
            ],
            nullable=False
        ),

        # Key numeric columns
        "LotArea": Column(
            dtype=float,
            checks=Check.greater_than(0),
            nullable=False
        ),

        "GrLivArea": Column(
            dtype=float,
            checks=Check.greater_than(0),
            nullable=False
        ),

        "OverallQual": Column(
            dtype=float,
            checks=[
                Check.greater_than_or_equal_to(1),
                Check.less_than_or_equal_to(10)
            ],
            nullable=False
        ),

        "YrSold": Column(
            dtype=int,
            checks=[
                Check.greater_than_or_equal_to(2006),
                Check.less_than_or_equal_to(2010)
            ],
            nullable=False
        ),
    },
    checks=[
        # Dataset level checks
        Check(lambda df: df.shape[0] == 1460, error="Row count must be 1460"),
        Check(lambda df: df.isnull().sum().sum() == 0, error="No nulls allowed"),
    ],
    coerce=True  # auto convert compatible types
)

print("Schema defined successfully!")

Schema defined successfully!


**Run Validation:**

In [13]:
try:
    validated_df = schema.validate(df, lazy=True)
    print("=" * 50)
    print("PANDERA VALIDATION PASSED!")
    print("=" * 50)
    print(f"All {len(schema.columns)} column contracts satisfied")
    print(f"Zero null values confirmed")
    print(f"All statistical boundaries respected")
    print(f"Dataset is ML-ready!")

except pa.errors.SchemaErrors as err:
    print("VALIDATION FAILED — issues found:")
    print(err.failure_cases)

PANDERA VALIDATION PASSED!
All 9 column contracts satisfied
Zero null values confirmed
All statistical boundaries respected
Dataset is ML-ready!


In [14]:
# Save final validated dataset
validated_df.to_csv('validated_clean_train.csv', index=False)
print("Validated dataset saved!")

Validated dataset saved!


In [16]:
# price_per_sqft feature (mentioned in project brief)
df['price_per_sqft'] = df['SalePrice'] / df['GrLivArea']
print(df['price_per_sqft'].describe())

count    1460.000000
mean      121.013462
std        31.918026
min        30.372058
25%       100.251375
50%       120.142243
75%       138.866679
max       276.250881
Name: price_per_sqft, dtype: float64


In [17]:
from sklearn.impute import KNNImputer

# Demonstrate KNN imputation knowledge
# (used for columns with >20% missing in enterprise scenarios)
imputer = KNNImputer(n_neighbors=5)
print("KNN Imputer ready — used for >20% missing columns")
print("Complexity: O(N^2) — computationally expensive but captures")
print("multi-dimensional relationships between features")

KNN Imputer ready — used for >20% missing columns
Complexity: O(N^2) — computationally expensive but captures
multi-dimensional relationships between features
